# KK1 – Analys av hundraser

Detta dataset beskriver **117 hundraser**, en rad per ras, med 8 kolumner: ras, ursprungsland, pälsfärg, höjd, ögonfärg, livslängd, karaktärsdrag och vanliga hälsoproblem.

**Källa:** Kaggle – https://www.kaggle.com/datasets/marshuu/dog-breeds

Höjden anges i **tum** i källan (kolumnen `Height (in)`) och livslängden i år, båda som textintervall (t.ex. `"21-24"`). Vi parsar dem till siffror och **räknar om höjden till centimeter**, eftersom rapporten är på svenska.

I notebooken läser vi in datan, inspekterar den, tvättar de två intervallkolumnerna och utforskar mönster med tre visualiseringar.

## Inläsning och inspektion

Först en mekanisk översikt av datan – form, kolumner, datatyper och saknade värden – innan vi tvättar något.

In [12]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Läs in datasetet
df = pd.read_csv("data/dog_breeds.csv")
df.shape

(117, 8)

In [13]:
df.head()

,Breed,Country of Origin,Fur Color,Height (in),Color of Eyes,Longevity (yrs),Character Traits,Common Health Problems
0,Labrador Retriever,Canada,"Yellow, Black, Chocolate",21-24,Brown,10-12,"Loyal, friendly, intelligent, energetic, good-...","Hip dysplasia, obesity, ear infections"
1,German Shepherd,Germany,"Black, Tan",22-26,Brown,7-10,"Loyal, intelligent, protective, confident, tra...","Hip dysplasia, elbow dysplasia, pancreatitis"
2,Bulldog,England,"White, Red",12-16,Brown,8-10,"Loyal, calm, gentle, brave","Skin allergies, respiratory issues, obesity"
3,Poodle,France,"White, Black, Brown, Apricot",10-15,"Brown, Blue",12-15,"Intelligent, active, affectionate, hypoallergenic","Hip dysplasia, epilepsy, bladder stones"
4,Beagle,England,"White, Tan, Red, Lemon",13-15,Brown,12-15,"Curious, friendly, energetic, good-natured","Ear infections, hip dysplasia, epilepsy"


In [14]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 117 entries, 0 to 116
Data columns (total 8 columns):
 #   Column                  Non-Null Count  Dtype
---  ------                  --------------  -----
 0   Breed                   117 non-null    str  
 1   Country of Origin       117 non-null    str  
 2   Fur Color               117 non-null    str  
 3   Height (in)             117 non-null    str  
 4   Color of Eyes           117 non-null    str  
 5   Longevity (yrs)         117 non-null    str  
 6   Character Traits        117 non-null    str  
 7   Common Health Problems  117 non-null    str  
dtypes: str(8)
memory usage: 7.4 KB


In [15]:
df.describe(include="all")

,Breed,Country of Origin,Fur Color,Height (in),Color of Eyes,Longevity (yrs),Character Traits,Common Health Problems
count,117,117,117,117,117,117,117,117
unique,103,27,69,67,4,15,38,30
top,Australian Shepherd,England,White,10-12,Brown,12-15,"Intelligent, energetic, playful, good-natured","Dental problems, eye issues, skin allergies"
freq,3,24,9,8,108,44,50,59


**Observationer:**

- Datasetet har 117 rader och 8 kolumner.
- Alla kolumner läses in som text – ingen är numerisk ännu.
- `df.info()` visar inga saknade värden i någon kolumn.
- `Height (in)` och `Longevity (yrs)` är textintervall (t.ex. `"21-24"`) och måste parsas till siffror för att kunna plottas. Det gör vi i nästa sektion.

## Datatvätt

`Height (in)` och `Longevity (yrs)` är textintervall (t.ex. `"21-24"`) och går inte att plotta som de är. Vi tvättar dem medvetet:

- Först kontrollerar vi saknade värden i stället för att blint köra `dropna()`.
- Varje intervall ersätts med sin **mittpunkt** – ett enda tal som går att plotta. Mittpunkten döljer spridningen inom intervallet; vi återkommer till det i avslutningen.
- Höjden räknas om från **tum till centimeter** (× 2,54, avrundat till heltal) eftersom rapporten är på svenska.

In [ ]:
# Kontroll av saknade värden innan vi tvättar
df.isna().sum()

Breed                     0
Country of Origin         0
Fur Color                 0
Height (in)               0
Color of Eyes             0
Longevity (yrs)           0
Character Traits          0
Common Health Problems    0
dtype: int64

In [33]:
# Kontroll av dataformen innan vi tvättar
print(f"Antal rader och kolumner: {df.shape}")

Antal rader och kolumner: (117, 8)


In [ ]:
# Kontroll av datatyper innan vi tvättar
df.dtypes

Breed                     str
Country of Origin         str
Fur Color                 str
Height (in)               str
Color of Eyes             str
Longevity (yrs)           str
Character Traits          str
Common Health Problems    str
dtype: object

In [ ]:
def range_mitt(text):
    """Tar ett intervall som '21-24' och returnerar mittpunkten (22.5)."""
    low, high = text.split("-")
    return (int(low) + int(high)) / 2

# Höjd: tum -> cm, avrundat till heltal
df["Höjd_cm"] = (df["Height (in)"].apply(range_mitt) * 2.54).round(0).astype(int)

# Livslängd: mittpunkt i år
df["Livslängd_mitt"] = df["Longevity (yrs)"].apply(range_mitt)

df[["Breed", "Height (in)", "Höjd_cm", "Longevity (yrs)", "Livslängd_mitt"]].head()

In [ ]:
# Verifiera att de nya kolumnerna är numeriska och utan saknade värden
print(df[["Höjd_cm", "Livslängd_mitt"]].dtypes)
print("Saknade värden i nya kolumner:", int(df[["Höjd_cm", "Livslängd_mitt"]].isna().sum().sum()))

**Resultat av tvätten:**

- Inga saknade värden fanns, så inga rader behövde tas bort – ett medvetet konstaterande, inte ett blint `dropna()`.
- `Höjd_cm` och `Livslängd_mitt` är nu numeriska och kompletta (117 värden vardera).
- Datan är redo att utforskas med visualiseringar.

## Visualiseringar

Här utforskar vi datan med tre diagram. Först: vilka länder kommer flest hundraser i datasetet ifrån?

In [ ]:
antal = df["Country of Origin"].value_counts().head(10)

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(antal.index, antal.values)
ax.set_title("Antal hundraser per ursprungsland (topp 10)")
ax.set_xlabel("Ursprungsland")
ax.set_ylabel("Antal raser")
ax.tick_params(axis="x", rotation=45)
fig.tight_layout()
plt.show()

England sticker ut tydligt med 24 raser – ungefär en femtedel av datasetet – följt av Tyskland (13) och Frankrike (10). De flesta länderna i toppen är europeiska.

Hur länge lever raserna typiskt? Vi tittar på fördelningen av median-livslängden (mittpunkten av varje ras livslängdsintervall).

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(df["Livslängd_mitt"], bins=10, edgecolor="black")
ax.set_title("Fördelning av median-livslängd")
ax.set_xlabel("Median-livslängd (år)")
ax.set_ylabel("Antal raser")
fig.tight_layout()
plt.show()

De flesta raser har en median-livslängd på ungefär 11–14 år, med en tydlig topp kring 13 år. Få raser hamnar under 10 år. Spannet går från 7 till 16 år (medel 12,5).

Finns det något samband mellan en ras höjd och hur länge den lever? Vi ställer höjd (cm) mot median-livslängd (år).

In [ ]:
# Färglägg punkterna efter de fem vanligaste ursprungsländerna (resten = "Övriga").
# Land är kategoriskt – vi tittar visuellt om något land klumpar ihop sig.
topp_lander = df["Country of Origin"].value_counts().head(5).index.tolist()
land_grupp = df["Country of Origin"].where(df["Country of Origin"].isin(topp_lander), "Övriga")

rng = np.random.default_rng(0)  # fast frö => samma bild varje körning

fig, ax = plt.subplots(figsize=(8, 5))
for land in topp_lander + ["Övriga"]:
    mask = land_grupp == land
    n = int(mask.sum())
    jitter_h = rng.uniform(-0.3, 0.3, n)
    jitter_l = rng.uniform(-0.08, 0.08, n)
    ax.scatter(df.loc[mask, "Höjd_cm"] + jitter_h,
               df.loc[mask, "Livslängd_mitt"] + jitter_l,
               alpha=0.6, label=land)
ax.set_title("Höjd mot livslängd per ursprungsland")
ax.set_xlabel("Höjd (cm)")
ax.set_ylabel("Livslängd (år)")
# Dra in x-axeln nära datans intervall så tomrummet i sidled krymper
ax.set_xlim(df["Höjd_cm"].min() - 1, df["Höjd_cm"].max() + 1)
ax.legend(title="Ursprungsland", fontsize="small")
fig.tight_layout()
plt.show()

Punkterna är nu färgade efter de fem vanligaste ursprungsländerna (övriga slås ihop). Den negativa tendensen höjd↔livslängd (korrelation ≈ −0,62) finns kvar, men inget enskilt land klumpar tydligt ihop sig i en egen del av diagrammet. Med bara en handfull raser per land är sådana mönster ändå osäkra – land är kategoriskt och har ingen egen korrelation med höjd eller livslängd.

## Avslutning

**Mönster vi sett:**

- Datasetet domineras av europeiska ursprungsländer, särskilt England (24 av 117 raser).
- Median-livslängden ligger oftast kring 11–14 år, med en topp runt 13 år.
- Det finns en måttlig negativ koppling mellan höjd och livslängd (korrelation ≈ −0,62) – större hundar lever i snitt kortare.

**Vad datan inte kan svara på:**

- Datasetet är ett urval av 117 *kända* raser, inte alla hundraser och inte enskilda individer – det säger inget om hur vanliga raserna är.
- Vi ersatte varje intervall med mittpunkten, vilket döljer spridningen inom en ras (en ras med "10–16 år" behandlas som 13).
- Insamlingsmetod och representativitet är odokumenterade (Kaggle), så siffrorna bör ses som ungefärliga.
- Den negativa höjd–livslängd-kopplingen är ett samband, inte ett bevisat orsakssamband.